## EDA Summary (from `Explanatory Data Analysis(EDA).ipynb`)

**Dataset:** PaySim — 6,362,620 synthetic mobile money transactions, 11 columns.

**Key findings:**
1. **Severe class imbalance** — fraud is only 0.129% of transactions. Accuracy is not a valid metric; evaluation will use Precision, Recall, F1, and PR-AUC.
2. **Fraud occurs only in `TRANSFER` and `CASH_OUT`** transactions — zero fraud in `PAYMENT`, `CASH_IN`, or `DEBIT`. `type` is a strong feature.
3. **Fraud transactions are larger** — ~6-8x bigger than legitimate ones on average and at the median.
4. **Disproportionate financial impact** — fraud is 0.13% of transaction count but 1.05% of total transaction volume.
5. **Origin account is almost always fully emptied in fraud** (~98% vs ~57% for legitimate transactions) — the clearest behavioral fraud signal found, but it relies on `newbalanceOrig`, a leakage column.
6. **`isFlaggedFraud` is not a useful feature** — it's essentially always 0, even for real fraud cases (near-zero recall as a rule-based flag).
7. **No strong time pattern** — fraud count fluctuates noisily across all 744 steps (hours), with no clear clustering.

**Data leakage columns (excluded from modeling):** `oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`, `newbalanceDest` — these reflect post-transaction state, and fraud transactions get cancelled, so they'd leak the label. `isFlaggedFraud` is also excluded (simulator artifact, not a real-world available feature).

**ID columns dropped:** `nameOrig`, `nameDest` — high-cardinality identifiers with no predictive value.

**Features going into modeling:** `step`, `type` (encoded), `amount`.

In [82]:
import pandas as pd 

In [83]:
data = pd.read_csv("../data/PS_20174392719_1491204439457_log.csv")

In [84]:
data.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


# Feature Engeneering

In [85]:
data = data.drop(columns = ["nameOrig","nameDest"]) 

In [86]:
data.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,0
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,0
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1,0
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,0


In [87]:
data["amount_to_balance_ratio"] = data["amount"] / (data["oldbalanceOrg"] + 1)

In [88]:
x = data.drop(columns=["newbalanceOrig", "newbalanceDest", "isFlaggedFraud", "isFraud"])
y = data["isFraud"]

In [89]:
x.dtypes

step                         int64
type                        object
amount                     float64
oldbalanceOrg              float64
oldbalanceDest             float64
amount_to_balance_ratio    float64
dtype: object

In [90]:
import sys
!{sys.executable} -m pip install scikit-learn
!pip install scikit-learn

In [91]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split


encoder = ColumnTransformer(
    transformers=[
        ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"), ["type"])
    ],
    remainder="passthrough"
)

In [92]:
X_train,X_test,y_train,y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=42
)

In [93]:
x_train = encoder.fit_transform(X_train)
x_test = encoder.transform(X_test)

# Model

In [94]:
import sys
!{sys.executable} -m pip install lightgbm xgboost

In [95]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import classification_report, precision_recall_curve, auc, confusion_matrix

In [96]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
models = {
    "LogisticRegression": LogisticRegression(class_weight="balanced", random_state=42),
    "lightgbm": lgb.LGBMClassifier(is_unbalance=True, random_state=42),
    "xgboost": xgb.XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric="logloss")
}

In [97]:
results = {}

for name, model in models.items():
    print(f"--- Training {name} ---")
    model.fit(X_train_encoded, y_train)
    y_pred = model.predict(X_test_encoded)
    
    print(classification_report(y_test, y_pred))
    results[name] = classification_report(y_test, y_pred, output_dict=True)

--- Training LogisticRegression ---


/opt/anaconda3/envs/ml_course/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              precision    recall  f1-score   support

           0       1.00      0.90      0.94   1270881
           1       0.01      0.83      0.02      1643

    accuracy                           0.90   1272524
   macro avg       0.50      0.86      0.48   1272524
weighted avg       1.00      0.90      0.94   1272524

--- Training lightgbm ---
[LightGBM] [Info] Number of positive: 6570, number of negative: 5083526
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017174 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1283
[LightGBM] [Info] Number of data points in the train set: 5090096, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001291 -> initscore=-6.651247
[LightGBM] [Info] Start training from score -6.651247
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [

In [98]:
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc, roc_auc_score

results = {}

for name, model in models.items():
    print(f"--- {name} ---")
    y_pred = model.predict(X_test_encoded)
    y_proba = model.predict_proba(X_test_encoded)[:, 1]
    
    # Classification report
    print(classification_report(y_test, y_pred))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix:")
    print(cm)
    
    # PR-AUC
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(recall, precision)
    
    # ROC-AUC (for comparison)
    roc_auc = roc_auc_score(y_test, y_proba)
    
    print(f"PR-AUC: {pr_auc:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}")
    print()
    
    results[name] = {"pr_auc": pr_auc, "roc_auc": roc_auc, "confusion_matrix": cm}

--- LogisticRegression ---
              precision    recall  f1-score   support

           0       1.00      0.90      0.94   1270881
           1       0.01      0.83      0.02      1643

    accuracy                           0.90   1272524
   macro avg       0.50      0.86      0.48   1272524
weighted avg       1.00      0.90      0.94   1272524

Confusion Matrix:
[[1137935  132946]
 [    276    1367]]
PR-AUC: 0.0380
ROC-AUC: 0.9416

--- lightgbm ---
              precision    recall  f1-score   support

           0       1.00      0.92      0.96   1270881
           1       0.01      0.66      0.02      1643

    accuracy                           0.92   1272524
   macro avg       0.51      0.79      0.49   1272524
weighted avg       1.00      0.92      0.96   1272524

Confusion Matrix:
[[1169296  101585]
 [    558    1085]]
PR-AUC: 0.3355
ROC-AUC: 0.7882

--- xgboost ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881

## Modeling Results

Four models were trained and evaluated on held-out test data (20% stratified split), using Precision, Recall, F1, PR-AUC, and ROC-AUC rather than accuracy (per the imbalance finding in the EDA).

| Model | Precision (Fraud) | Recall (Fraud) | F1 (Fraud) | PR-AUC | ROC-AUC |
|---|---|---|---|---|---|
| Logistic Regression | 0.01 | 0.83 | 0.02 | 0.038 | 0.942 |
| LightGBM | 0.01 | 0.66 | 0.02 | 0.336 | 0.788 |
| XGBoost | **0.64** | **0.98** | **0.77** | **0.968** | **0.9997** |

### Key Takeaways

- **XGBoost is the best-performing model**, catching ~98% of fraud cases with 64% precision — only 26 missed frauds and 917 false alarms out of 1.27M test transactions.
- **ROC-AUC is misleading under this level of imbalance.** Logistic Regression scores a deceptively high ROC-AUC (0.94) despite being nearly unusable in practice (132,946 false positives). PR-AUC (0.038) correctly exposes this — reinforcing the EDA's accuracy-is-misleading finding at the model evaluation stage too.
- **LightGBM underperformed its potential** — training logs showed repeated "no further splits with positive gain" warnings, indicating its default hyperparameters (e.g. `min_child_samples`) are too conservative for a 0.13% positive class. Its PR-AUC (0.336) suggests the underlying model has more signal than its default-threshold classification report shows — worth tuning if revisited.

### Next Steps / Limitations
- Check train-vs-test performance for overfitting
- This is a synthetic dataset (PaySim) — some separability (e.g. fraud confined to 2 transaction types, suspicious $10M cap) may be simulator artifacts rather than real-world fraud patterns
- No cross-validation performed yet — single train/test split

In [99]:
import joblib

joblib.dump(models["xgboost"], "xgboost_fraud_model.pkl")

['xgboost_fraud_model.pkl']